# Inheritance, composition, and the MRO

This module is about choosing the right relationship between types. **Composition** means one object contains another and delegates work to it. **Inheritance** means one type promises to behave like a more general type. Those are not interchangeable decisions.

Python adds another layer of complexity through the Method Resolution Order, or MRO. In multiple inheritance, `super()` does not simply mean “call my parent”. It means “continue to the next class in the MRO for this instance”. That rule is powerful, but many developers use it without fully understanding it.

As you study these exercises, keep testing the behavioural question: **should this design be an is-a relationship, or a has-a relationship?** That is often the difference between a clean hierarchy and a fragile one.

## Visual model

```text
D -> B -> C -> A -> object
super() follows THIS order, not just the direct parent
```

## How to use this notebook

Read the concept notes first, then run the code cells one at a time. After each run, change an input, prediction, or line of code and rerun it. Intermediate Python becomes easier when you treat every notebook as a place to test a mental model, not just a place to read finished answers.

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.


---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. `super()` and the MRO

In [ ]:
class A:
    def greet(self) -> str: return "A"

class B(A):
    def greet(self) -> str: return "B -> " + super().greet()

class C(A):
    def greet(self) -> str: return "C -> " + super().greet()

class D(B, C):
    def greet(self) -> str: return "D -> " + super().greet()

D().greet()          # 'D -> B -> C -> A'
D.__mro__            # (D, B, C, A, object)

**Look at `B.greet` again.** Its `super()` call went to `C`, not to `A` — even
though `B`'s only base is `A`, and `C` is nowhere in `B`'s definition. That is
the whole lesson:

> `super()` follows the MRO **of the instance's type**, not the class hierarchy
> written in the file.

This is why the mental model "super means my parent" is not merely imprecise but
actively wrong, and why cooperative multiple inheritance works at all.

### C3 linearisation

The MRO is computed by the C3 algorithm, which guarantees:

1. A class appears before all of its bases.
2. Bases appear in the order written.
3. The order is **monotonic** — a class's MRO is consistent with the MRO of
   every subclass.

If no order satisfies all three, the class statement raises at definition time:

In [ ]:
class X: pass
class Y(X): pass
class Z(X, Y): pass
# TypeError: Cannot create a consistent method resolution order (MRO)

That is Python refusing to build a hierarchy whose behaviour would be
ambiguous — a much better outcome than resolving it arbitrarily.

Read `SomeClass.__mro__` any time you are unsure. It is not something to
compute in your head.

### Cooperative inheritance

For multiple inheritance to work, **every** class in the chain must call
`super()`, and signatures must be compatible:

In [ ]:
class Base:
    def __init__(self, **kwargs) -> None:
        super().__init__(**kwargs)          # reaches object at the end

class Loggable:
    def __init__(self, *, log_level="INFO", **kwargs) -> None:
        self.log_level = log_level
        super().__init__(**kwargs)          # pass the rest ON

class Serializable:
    def __init__(self, *, fmt="json", **kwargs) -> None:
        self.fmt = fmt
        super().__init__(**kwargs)

class Widget(Loggable, Serializable, Base):
    def __init__(self, name: str, **kwargs) -> None:
        self.name = name
        super().__init__(**kwargs)

Widget("w", log_level="DEBUG", fmt="xml")

The `**kwargs` threading is what makes this work: each class consumes its own
keywords and passes the rest along. One class forgetting `super().__init__()`
silently breaks every class after it in the MRO — and "after it" is not visible
from that class's own source.

That fragility is the practical argument for keeping multiple inheritance to
**stateless mixins** (see below).

---

## Concept 3. Mixins

A mixin adds behaviour to a class it knows nothing about.

In [ ]:
class ReprMixin:
    """Adds a useful __repr__ to any class with a __dict__."""
    def __repr__(self) -> str:
        args = ", ".join(f"{k}={v!r}" for k, v in vars(self).items())
        return f"{type(self).__name__}({args})"

class ComparableMixin:
    """Adds ordering from a _key() method the host class provides."""
    def _key(self):
        raise NotImplementedError
    def __lt__(self, other):
        if not isinstance(other, type(self)):
            return NotImplemented
        return self._key() < other._key()

class Product(ReprMixin, ComparableMixin):
    def __init__(self, name: str, price: float) -> None:
        self.name, self.price = name, price
    def _key(self):
        return (self.price, self.name)

Rules that keep mixins from becoming a maze:

- **Mixins first in the bases list.** They must come before the concrete base in
  the MRO to be able to override it.
- **No state, or as little as possible.** A stateful mixin needs `__init__`
  cooperation, which is where multiple inheritance gets fragile.
- **Name them `-Mixin`.** The name is documentation.
- **Never instantiate one directly.** It is not a complete type.
- **Depend on a small, documented interface** (here: `_key`). Say what the host
  class must provide.

---

## Concept 4. Abstract base classes

In [ ]:
from abc import ABC, abstractmethod

class Storage(ABC):
    @abstractmethod
    def read(self, key: str) -> bytes: ...

    @abstractmethod
    def write(self, key: str, value: bytes) -> None: ...

    def read_text(self, key: str, encoding: str = "utf-8") -> str:
        return self.read(key).decode(encoding)      # a CONCRETE helper

class MemoryStorage(Storage):
    def __init__(self) -> None:
        self._data: dict[str, bytes] = {}
    def read(self, key: str) -> bytes:
        return self._data[key]
    def write(self, key: str, value: bytes) -> None:
        self._data[key] = value

Storage()          # TypeError: Can't instantiate abstract class

ABCs give you: instantiation refused until every abstract method is implemented,
shared concrete helpers, and `isinstance` working. The cost is that
implementations must **inherit** — which is fine inside your own codebase and a
real imposition on third-party types you do not control.

---

## Concept 5. `Protocol`: interfaces without inheritance

In [ ]:
from typing import Protocol, runtime_checkable

class SupportsRead(Protocol):
    def read(self, key: str) -> bytes: ...

def load(storage: SupportsRead, key: str) -> bytes:
    return storage.read(key)

Any object with a matching `read` satisfies this — **no inheritance, no
registration, nothing imported by the implementer.** This is static duck typing:
the type checker verifies structurally, at compile time, what Python was already
doing dynamically at runtime.

| | ABC | Protocol |
|---|---|---|
| Implementer must inherit | Yes | No |
| Works with third-party types | No | Yes |
| Checked | At instantiation, runtime | By mypy, statically |
| `isinstance` | Always | Only with `@runtime_checkable`, and it checks method *names* only |
| Can provide implementations | Yes | Only defaults (3.8+), rarely used |

**Rule of thumb:** `Protocol` for describing what you *accept*; ABC for a family
of types you *own* and want to share code between. When in doubt, `Protocol` —
it couples less.

Note the `runtime_checkable` limitation: `isinstance` against a protocol checks
only that the attribute *names* exist, not their signatures. It is a weak check
and should not be relied on for correctness.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: Is-a versus has-a
- Section 2: `super()` and the MRO
- Section 3: Mixins
- Section 4: Abstract base classes
- Section 5: `Protocol`: interfaces without inheritance
- Section 6: Duck typing, and when to stop `isinstance`

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

from abc import ABC, abstractmethod
from typing import Protocol, runtime_checkable

# --- the third-party class you do NOT control ---------------------------------

---

## `VendorSlugifier`

Pretend this comes from a library you cannot modify. Note it already has

In [ ]:
class VendorSlugifier:
    """Pretend this comes from a library you cannot modify. Note it already has
    exactly the right method -- it just has never heard of you."""

    def transform(self, text: str) -> str:
        return text.lower().replace(" ", "-")

---

## `TransformABC`

Abstract `transform`, plus a concrete `apply_twice` helper.

In [ ]:
class TransformABC(ABC):
    """Abstract `transform`, plus a concrete `apply_twice` helper."""

---

## `TransformProto`

Structural. Implementations do not import this.

In [ ]:
class TransformProto(Protocol):
    """Structural. Implementations do not import this."""

---

## `run_pipeline`

Apply every step in order. Type the `steps` parameter three ways and see

In [ ]:
def run_pipeline(text: str, steps: list) -> str:  # type: ignore[type-arg]
    """Apply every step in order. Type the `steps` parameter three ways and see
    which the type checker accepts."""
    raise NotImplementedError

---

## `verify`

_verify_

In [ ]:
def verify() -> None:
    steps_abc = [Upper(), Reverse()]                 # type: ignore[name-defined]
    assert run_pipeline("abc", steps_abc) == "CBA"

    steps_proto = [UpperProto(), ReverseProto()]     # type: ignore[name-defined]
    assert run_pipeline("abc", steps_proto) == "CBA"

    steps_mixed = [UpperProto(), VendorSlugifier()]  # type: ignore[name-defined]
    assert run_pipeline("Hello World", steps_mixed) == "hello-world"

    print("all pipeline checks passed")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    verify()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.